# 03.5 Data Preparation (YOLO)

1. 전처리 이미지 → `yolo/{pipe}/images/{train,val}`
2. original(raw) → `yolo/original/images/{train,val}`
3. COCO stenosis(26) → YOLO labels class 0
4. 개수 확인


In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path.cwd().parent / 'data'

ORIG_TRAIN_DIR = DATA_DIR / 'arcade/arcade/stenosis/train/images'
ORIG_VAL_DIR = DATA_DIR / 'arcade/arcade/stenosis/val/images'

TRAIN_JSON = DATA_DIR / 'arcade/arcade/stenosis/train/annotations/train.json'
VAL_JSON = DATA_DIR / 'arcade/arcade/stenosis/val/annotations/val.json'

PREPROCESS_DIR = DATA_DIR / 'preprocessed'
YOLO_DIR = DATA_DIR / 'yolo'

pipelines = {
    '01_clahe': 'clahe',
    '02_gamma': 'gamma',
    '03_frangi': 'frangi',
    '04_clahe_gamma': 'clahe_gamma',
    '05_clahe_gamma_frangi': 'clahe_gamma_frangi',
    '06_gamma_clahe_frangi': 'gamma_clahe_frangi',
    '07_black_hat': 'black_hat',
}

TARGET_COCO_ID = 26
YOLO_CLASS_ID = 0

print('TRAIN_JSON exists:', TRAIN_JSON.exists())
print('VAL_JSON exists  :', VAL_JSON.exists())
print('PREPROCESS_DIR   :', PREPROCESS_DIR.exists())


## 1) 전처리 이미지 → YOLO images


In [ ]:
# Copy preprocessed images by split
# preprocessed/01_*/{train,val} -> yolo/{pipe}/images/{train,val}

for prep_folder, yolo_folder in pipelines.items():
    print(f'\n[{prep_folder}] -> [{yolo_folder}]')

    src_train = PREPROCESS_DIR / prep_folder / 'train'
    src_val = PREPROCESS_DIR / prep_folder / 'val'

    if not src_train.exists() or not src_val.exists():
        print(f'  missing source split dirs train={src_train.exists()} val={src_val.exists()}')
        continue

    dst_train = YOLO_DIR / yolo_folder / 'images' / 'train'
    dst_val = YOLO_DIR / yolo_folder / 'images' / 'val'
    dst_train.mkdir(parents=True, exist_ok=True)
    dst_val.mkdir(parents=True, exist_ok=True)

    n_train = 0
    for img_path in tqdm(list(src_train.glob('*.png')), desc=f'{yolo_folder}/train'):
        shutil.copy2(img_path, dst_train / img_path.name)
        n_train += 1

    n_val = 0
    for img_path in tqdm(list(src_val.glob('*.png')), desc=f'{yolo_folder}/val'):
        shutil.copy2(img_path, dst_val / img_path.name)
        n_val += 1

    print(f'  copied train={n_train}, val={n_val}')

print('copy done')


## 2) original(raw) → YOLO images


In [3]:
# original baseline: raw train/val 복사

orig_train_dst = YOLO_DIR / "original" / "images" / "train"
orig_val_dst = YOLO_DIR / "original" / "images" / "val"
orig_train_dst.mkdir(parents=True, exist_ok=True)
orig_val_dst.mkdir(parents=True, exist_ok=True)

for src_path in tqdm(list(ORIG_TRAIN_DIR.glob("*.png")), desc="original/train"):
    shutil.copy2(src_path, orig_train_dst / src_path.name)

for src_path in tqdm(list(ORIG_VAL_DIR.glob("*.png")), desc="original/val"):
    shutil.copy2(src_path, orig_val_dst / src_path.name)

print("original train:", len(list(orig_train_dst.glob("*.png"))))
print("original val  :", len(list(orig_val_dst.glob("*.png"))))


original train: 1000
original val  : 200


## 3) COCO → YOLO labels (stenosis only)

- COCO `category_id == 26` 만 사용
- YOLO class id = `0`
- 라벨은 전처리와 무관하므로 모든 파이프라인(+original)에 동일 내용 복사


In [4]:
def convert_coco_to_yolo_labels(json_path: Path):
    """file_name -> list[yolo_str]"""
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    img_dict = {
        img["id"]: {
            "file_name": img["file_name"],
            "w": img["width"],
            "h": img["height"],
        }
        for img in coco["images"]
    }

    yolo_labels = {img["file_name"]: [] for img in coco["images"]}
    n_boxes = 0

    for ann in coco["annotations"]:
        if ann.get("category_id") != TARGET_COCO_ID:
            continue

        info = img_dict[ann["image_id"]]
        file_name = info["file_name"]
        img_w, img_h = info["w"], info["h"]

        x_min, y_min, bbox_w, bbox_h = ann["bbox"]
        x_center = (x_min + bbox_w / 2.0) / img_w
        y_center = (y_min + bbox_h / 2.0) / img_h
        norm_w = bbox_w / img_w
        norm_h = bbox_h / img_h

        # clip to [0, 1]
        x_center = min(max(x_center, 0.0), 1.0)
        y_center = min(max(y_center, 0.0), 1.0)
        norm_w = min(max(norm_w, 0.0), 1.0)
        norm_h = min(max(norm_h, 0.0), 1.0)

        yolo_str = (
            f"{YOLO_CLASS_ID} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )
        yolo_labels[file_name].append(yolo_str)
        n_boxes += 1

    n_empty = sum(1 for v in yolo_labels.values() if len(v) == 0)
    print(f"  images={len(yolo_labels)}, stenosis_boxes={n_boxes}, empty_labels={n_empty}")
    return yolo_labels


def write_labels_for_pipelines(yolo_labels: dict, split_name: str, yolo_folders: list):
    for pipe in yolo_folders:
        label_dir = YOLO_DIR / pipe / "labels" / split_name
        label_dir.mkdir(parents=True, exist_ok=True)
        for file_name, labels in yolo_labels.items():
            txt_path = label_dir / (Path(file_name).stem + ".txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write("\n".join(labels))


# YOLO 폴더 목록: pipelines values + original
yolo_folders = list(pipelines.values()) + ["original"]

print("[train.json]")
train_labels = convert_coco_to_yolo_labels(TRAIN_JSON)
write_labels_for_pipelines(train_labels, "train", yolo_folders)

print("[val.json]")
val_labels = convert_coco_to_yolo_labels(VAL_JSON)
write_labels_for_pipelines(val_labels, "val", yolo_folders)

print("라벨 변환/복사 완료")


[train.json]
  images=1000, stenosis_boxes=1625, empty_labels=3
[val.json]
  images=200, stenosis_boxes=406, empty_labels=0
라벨 변환/복사 완료


## 4) 개수 확인 (images vs labels)


In [5]:
def count_png(p: Path) -> int:
    return len(list(p.glob("*.png"))) if p.exists() else 0

def count_txt(p: Path) -> int:
    return len(list(p.glob("*.txt"))) if p.exists() else 0

all_pipes = list(pipelines.values()) + ["original"]

print(f"{'pipe':25s} {'img_tr':>7} {'lbl_tr':>7} {'img_va':>7} {'lbl_va':>7}  ok?")
print("-" * 70)

for pipe in all_pipes:
    img_tr = count_png(YOLO_DIR / pipe / "images" / "train")
    img_va = count_png(YOLO_DIR / pipe / "images" / "val")
    lbl_tr = count_txt(YOLO_DIR / pipe / "labels" / "train")
    lbl_va = count_txt(YOLO_DIR / pipe / "labels" / "val")
    ok = (img_tr == lbl_tr and img_va == lbl_va and img_tr > 0 and img_va > 0)
    print(
        f"{pipe:25s} {img_tr:7d} {lbl_tr:7d} {img_va:7d} {lbl_va:7d}  "
        f"{'OK' if ok else 'CHECK'}"
    )


pipe                       img_tr  lbl_tr  img_va  lbl_va  ok?
----------------------------------------------------------------------
clahe                        1000    1000       0     200  CHECK
gamma                        1000    1000       0     200  CHECK
frangi                       1000    1000       0     200  CHECK
clahe_gamma                  1000    1000       0     200  CHECK
clahe_gamma_frangi           1000    1000       0     200  CHECK
gamma_clahe_frangi           1000    1000       0     200  CHECK
original                     1000    1000     200     200  OK
